# Homework 2 - Machine Learning for Regression


## Setup


In [2]:
import pandas as pd
import numpy as np

## Getting the data


In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Preparing the dataset

Use only the following columns:
* 'engine_displacement'
* 'horsepower'
* 'vehicle_weight'
* 'model_year'
* 'fuel_efficiency_mpg'


In [4]:
cols = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]
df = df[cols]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


## EDA

Look at the fuel_efficiency_mpg variable. Does it have a long tail?


In [5]:
df['fuel_efficiency_mpg'].describe()

count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64

## Question 1


In [6]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

## Question 2


In [7]:
df['horsepower'].median()

np.float64(254.0)

## Prepare and split the dataset


In [16]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]


def rmse(y, y_pred):
    error = y - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)


def prepare_X(df, fill_value):
    df = df.copy()
    df['horsepower'] = df['horsepower'].fillna(fill_value)
    X = df[['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']].values
    return X

In [17]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

In [18]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

## Question 3


In [19]:
# fill with 0
X_train = prepare_X(df_train, 0)
w0, w = train_linear_regression(X_train, y_train)
X_val = prepare_X(df_val, 0)
y_pred = w0 + X_val.dot(w)
print('RMSE with 0:', round(rmse(y_val, y_pred), 3))

# fill with mean
mean_hp = df_train['horsepower'].mean()
X_train = prepare_X(df_train, mean_hp)
w0, w = train_linear_regression(X_train, y_train)
X_val = prepare_X(df_val, mean_hp)
y_pred = w0 + X_val.dot(w)
print('RMSE with mean:', round(rmse(y_val, y_pred), 3))

RMSE with 0: 2.205
RMSE with mean: 2.202


## Question 4


In [20]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare_X(df_train, 0)
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    X_val = prepare_X(df_val, 0)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    print(f'r={r}: {round(score, 4)}')

r=0: 2.2053
r=0.01: 2.2058
r=0.1: 2.2241
r=1: 2.3492
r=5: 2.4094
r=10: 2.4195
r=100: 2.4292


## Question 5


In [22]:
scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_tr = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_v = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)

    y_tr = df_tr['fuel_efficiency_mpg'].values
    y_v = df_v['fuel_efficiency_mpg'].values

    X_tr = prepare_X(df_tr, 0)
    w0, w = train_linear_regression(X_tr, y_tr)
    X_v = prepare_X(df_v, 0)
    y_pred = w0 + X_v.dot(w)
    score = rmse(y_v, y_pred)
    scores.append(score)
    print(f'seed={seed}: {score:.6f}')

print('std:', round(np.std(scores), 3))

seed=0: 2.239204
seed=1: 2.202113
seed=2: 2.163420
seed=3: 2.204359
seed=4: 2.201880
seed=5: 2.245785
seed=6: 2.269721
seed=7: 2.190795
seed=8: 2.216611
seed=9: 2.207037
std: 0.029


## Question 6


In [23]:
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_tr = df.iloc[idx[:n_train]].reset_index(drop=True)
df_v = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_te = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

df_full = pd.concat([df_tr, df_v]).reset_index(drop=True)
y_full = df_full['fuel_efficiency_mpg'].values
y_te = df_te['fuel_efficiency_mpg'].values

X_full = prepare_X(df_full, 0)
w0, w = train_linear_regression_reg(X_full, y_full, r=0.001)

X_te = prepare_X(df_te, 0)
y_pred = w0 + X_te.dot(w)
print(round(rmse(y_te, y_pred), 3))

2.236
